### **Installations and Imports**

In [1]:
import os

# Prevent rare Unsloth compile / patch issues on some setups
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"


import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil   = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil   = "pillow"

    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0

# causal_conv1d is supported only on torch==2.8.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.4 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Resolved 4 packages in 30ms
Prepared 1 package in 36ms
Uninstalled 1 package in 1ms
Installed 1 package in 5ms
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.12 environment at: /usr
Audited 1 package in 97ms
Using Python 3.12.12 environment at: /usr
Resolved 55 packages in 145ms
Prepared 4 packages in 13.13s
Installed 4 packages in 9ms
 + causal-conv1d==1.6.0
 + fla-core==0.4.1
 + flash-linear-attention==0.4.1
 + ninja==1.13.0


### **Load Qwen3.5-2B Vision**

In [2]:
import torch
from unsloth import FastVisionModel

print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "unsloth/Qwen3.5-2B"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit = False,
    dtype = torch.float32,
    use_gradient_checkpointing = "unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
==((====))==  Unsloth 2026.3.3: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

### **Attach LoRA**

In [3]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model.visual` require gradients


### force Vision tower + LayerNorm to fp32

In [4]:
import torch.nn as nn

def force_fp32_everywhere(m):
    m = m.to(torch.float32)

    # Force all LayerNorm to fp32 (common source of Half/Float mismatch)
    for mod in m.modules():
        if isinstance(mod, nn.LayerNorm):
            mod.to(torch.float32)

    # Try common paths to the vision tower and force fp32
    candidate_paths = [
        "model.visual",
        "base_model.model.visual",
        "model.model.visual",
        "base_model.model.model.visual",
    ]
    for path in candidate_paths:
        obj = m
        ok = True
        for part in path.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            obj.to(torch.float32)

    return m

model = force_fp32_everywhere(model)

### **Loading Arabic OCR dataset and building conversations**

In [5]:
from datasets import load_dataset, Image as HFImage

DATASET_NAME = "mssqpi/Arabic-OCR-Dataset"

raw = load_dataset(DATASET_NAME, split="train")
raw = raw.cast_column("image", HFImage(decode=True))
raw = raw.shuffle(seed=3407)

MAX_TRAIN_SAMPLES = 20000
raw = raw.select(range(min(MAX_TRAIN_SAMPLES, len(raw))))

TEXT_COL = "text" if "text" in raw.column_names else raw.column_names[0]  # fallback

instruction = "Read the Arabic text in this image exactly. Output ONLY the text."

def to_conversation(ex):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text",  "text": instruction},
                    {"type": "image", "image": ex["image"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": ex[TEXT_COL]},
                ],
            },
        ]
    }

# ✅ Use a Python list like the official Unsloth notebook (avoids the ['messages'] padding issue)
train_ds = [to_conversation(raw[i]) for i in range(len(raw))]
print("train_ds[0] keys:", train_ds[0].keys(), "| n =", len(train_ds))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/471M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/475M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/427M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/494M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2160000 [00:00<?, ? examples/s]

train_ds[0] keys: dict_keys(['messages']) | n = 20000


### Sanity check collator output

In [6]:
from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(model, tokenizer)
batch = collator([train_ds[0], train_ds[1]])

print("Batch keys:", list(batch.keys()))
print("input_ids shape:", batch["input_ids"].shape)
print("pixel_values dtype:", batch["pixel_values"].dtype)

# Check a LayerNorm dtype (should be float32 now)
import torch.nn as nn
ln = next((m for m in model.modules() if isinstance(m, nn.LayerNorm)), None)
print("First LayerNorm weight dtype:", None if ln is None else ln.weight.dtype)

Unsloth: Model does not have a default image size - using 512
Batch keys: ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'labels']
input_ids shape: torch.Size([2, 104])
pixel_values dtype: torch.float32
First LayerNorm weight dtype: torch.float32


### **Train with SFTTrainer and UnslothVisionDataCollator**

In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)
model = force_fp32_everywhere(model)   # Important

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        #num_train_epochs=1,
        max_steps = 500,
        learning_rate = 2e-4,
        warmup_steps = 100,
        logging_steps = 10,

        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # avoid any fp16/bf16 autocast
        fp16 = False,
        bf16 = False,

        # MUST for vision finetuning (as in Unsloth notebook)
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

trainer_stats = trainer.train()
trainer_stats

Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 23,110,656 of 2,236,352,320 (1.03% trained)


Step,Training Loss
10,9.785966
20,6.366412
30,1.954728
40,0.526562
50,0.491875
60,0.490424
70,0.542078
80,0.483307
90,0.413207
100,0.426201


Step,Training Loss
10,9.785966
20,6.366412
30,1.954728
40,0.526562
50,0.491875
60,0.490424
70,0.542078
80,0.483307
90,0.413207
100,0.426201


TrainOutput(global_step=500, training_loss=0.6508693063259124, metrics={'train_runtime': 3721.2575, 'train_samples_per_second': 1.075, 'train_steps_per_second': 0.134, 'total_flos': 4304135685255936.0, 'train_loss': 0.6508693063259124, 'epoch': 0.2})

### **Save LoRA adapter**

In [8]:
SAVE_DIR = "qwen3_5_2b_arabic_ocr_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

Saved to: qwen3_5_2b_arabic_ocr_lora


### **Inference on a random sample**

In [9]:
import random
from transformers import TextStreamer

FastVisionModel.for_inference(model)

idx = random.randint(0, len(raw) - 1)
image = raw[idx]["image"]
gt = raw[idx][TEXT_COL]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction},
    ]}
]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.0,
        use_cache=True,
    )

gen = out[0, inputs["input_ids"].shape[1]:]
pred = tokenizer.decode(gen, skip_special_tokens=True).strip()

print("GT  :", gt)
print("PRED:", pred)

GT  : انجابها
PRED: انجابها


### **Evaluation**

In [13]:
# Reuse the already-loaded dataset (no re-download)
from datasets import Image as HFImage

TRAIN_SIZE = 20000
EVAL_SIZE  = 1000
SEED = 3407

# Use raw_full if you have it; otherwise fall back to raw
ds = raw_full if "raw_full" in globals() else raw

# Ensure images are decoded as PIL (safe even if already decoded)
if "image" in ds.column_names:
    ds = ds.cast_column("image", HFImage(decode=True))

# Shuffle once for a clean split
ds = ds.shuffle(seed=SEED)
n = len(ds)

if n >= TRAIN_SIZE + EVAL_SIZE:
    # Proper holdout AFTER the training slice (clean, no leakage)
    train_raw = ds.select(range(TRAIN_SIZE))
    eval_raw  = ds.select(range(TRAIN_SIZE, TRAIN_SIZE + EVAL_SIZE))
else:

    # (NOTE: if you already trained on all of ds, this eval is NOT a true holdout)
    eval_size = min(EVAL_SIZE, n // 20 if n >= 2000 else max(1, n // 10))
    eval_raw  = ds.select(range(eval_size))
    train_raw = ds.select(range(eval_size, n))

print("Total:", n)
print("Train size:", len(train_raw))
print("Eval size :", len(eval_raw))
print("Eval example GT:", eval_raw[0]["text"])

Total: 20000
Train size: 19000
Eval size : 1000
Eval example GT: كالطائف


### **Arabic text normalization (for normalized CER/WER)**

In [14]:
import re

_AR_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")  # harakat + Quran marks
_TATWEEL = "\u0640"

DIGIT_MAP = str.maketrans({
    "٠":"0","١":"1","٢":"2","٣":"3","٤":"4","٥":"5","٦":"6","٧":"7","٨":"8","٩":"9",
    "۰":"0","۱":"1","۲":"2","۳":"3","۴":"4","۵":"5","۶":"6","۷":"7","۸":"8","۹":"9",
})

def normalize_arabic(s: str) -> str:
    s = s.strip()
    s = s.translate(DIGIT_MAP)
    s = s.replace(_TATWEEL, "")
    s = _AR_DIACRITICS.sub("", s)

    # common letter normalization (light)
    s = re.sub("[إأآٱ]", "ا", s)
    s = s.replace("ى", "ي")

    # normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

### **Batched inference**

In [15]:
import torch
from tqdm.auto import tqdm

FastVisionModel.for_inference(model)

@torch.no_grad()
def predict_batch(images, prompt_text, max_new_tokens=128):
    # Unsloth inference template: image placeholder + text instruction
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt_text},
    ]}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    texts = [prompt] * len(images)
    inputs = tokenizer(
        images,
        texts,
        add_special_tokens=False,
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        use_cache=True,
    )

    # decode only the generated continuation (strip the prompt tokens)
    input_lens = inputs["input_ids"].shape[1]
    gens = out[:, input_lens:]
    preds = [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gens]
    return preds

###**Run predictions**

In [16]:
BATCH_SIZE = 4
MAX_NEW_TOKENS = 128

gts = []
preds = []

for i in tqdm(range(0, len(eval_raw), BATCH_SIZE)):
    batch = eval_raw[i:i+BATCH_SIZE]
    images = batch["image"]
    gt_texts = batch["text"]

    batch_preds = predict_batch(images, instruction, max_new_tokens=MAX_NEW_TOKENS)

    gts.extend(gt_texts)
    preds.extend(batch_preds)

print("Done. Preds:", len(preds))
print("Sample pred:", preds[0])

  0%|          | 0/250 [00:00<?, ?it/s]

Done. Preds: 1000
Sample pred: كالطائف


### **Compute CER/WER , normalized CER/WER and exact match metrics**

In [17]:
!pip -q install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.2 MB/s eta 0:00:00


In [18]:
from jiwer import wer, cer

# raw metrics
raw_cer = cer(gts, preds)
raw_wer = wer(gts, preds)
exact_match = sum(int(g.strip() == p.strip()) for g, p in zip(gts, preds)) / len(gts)

# normalized metrics
gts_n  = [normalize_arabic(x) for x in gts]
preds_n = [normalize_arabic(x) for x in preds]

norm_cer = cer(gts_n, preds_n)
norm_wer = wer(gts_n, preds_n)
exact_match_norm = sum(int(g == p) for g, p in zip(gts_n, preds_n)) / len(gts_n)

print("RAW   CER:", raw_cer)
print("RAW   WER:", raw_wer)
print("RAW   Exact Match:", exact_match)

print("NORM  CER:", norm_cer)
print("NORM  WER:", norm_wer)
print("NORM  Exact Match:", exact_match_norm)

RAW   CER: 0.35812356979405036
RAW   WER: 0.72
RAW   Exact Match: 0.608
NORM  CER: 0.33524684270952926
NORM  WER: 0.725
NORM  Exact Match: 0.644


### **Error analysis (see worst cases)**

In [19]:
def per_sample_cer(a, b):
    return cer([a], [b])

scores = []
for gt, pr, gtn, prn in zip(gts, preds, gts_n, preds_n):
    scores.append(per_sample_cer(gtn, prn))

worst = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:10]
for i in worst:
    print("="*60)
    print("CER(norm):", scores[i])
    print("GT :", gts[i])
    print("PR :", preds[i])
    print("GTn:", gts_n[i])
    print("PRn:", preds_n[i])

CER(norm): 84.875
GT : والوقاية
PR : and the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the
GTn: والوقاية
PRn: and the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the location of the 